In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge, Lasso

In [32]:
salary_raw = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Educator_Salary.csv', low_memory=False)
mobility_raw = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Educator_Mobility.csv', low_memory=False)

In [33]:
display(mobility_raw.head())

,Unnamed: 0,School Year,District Code,District,School Code,Organization,Race,Gender,Grade,SpecialDemo,...,Retention Type,Same School Retention Rate,Transfer Rate Within District,Transfer Rate Between Districts,Turnover Rate,Same School Numerator,Transfer Within District Numerator,Transfer Between Districts Numerator,Turnover Numerator,Total Number of Staff
0,0,2023,0,State of Delaware,0,State of Delaware,African American,All Educators,All Educators,All Educators,...,Five Year Percentage,43.8,11.4,18.6,26.2,426,111,181,255,973
1,1,2023,0,State of Delaware,0,State of Delaware,African American,All Educators,All Educators,All Educators,...,Four Year Percentage,40.9,10.4,17.2,31.4,311,79,131,239,760
2,2,2023,0,State of Delaware,0,State of Delaware,African American,All Educators,All Educators,All Educators,...,One Year Percentage,79.5,3.8,8.5,8.3,931,44,100,97,"1,171"
3,3,2023,0,State of Delaware,0,State of Delaware,African American,All Educators,All Educators,All Educators,...,Three Year Percentage,58.6,9.0,14.7,17.7,656,101,165,198,"1,120"
4,4,2023,0,State of Delaware,0,State of Delaware,African American,All Educators,All Educators,All Educators,...,Two Year Percentage,66.4,7.2,11.9,14.4,773,84,139,168,"1,164"


In [34]:
salary_raw.head()

,Unnamed: 0,School Year,District Code,District,School Code,Organization,Race,Gender,Grade,SpecialDemo,...,Staff Category,Job Classification,Experience,Educators (FTE),Average Total Salary,Average State Salary,Average Local Salary,Average Federal Salary,Average Years of Experience,Average Years of Age
0,0,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,Regular,...,Official/Administrative,ALL,ALL,14,"95,625.22","54,023.65","41,601.56",NaN,19,53
1,1,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,Regular,...,Pupil Support,ALL,ALL,1,"97,917.55","60,056.88","37,860.68",NaN,1,33
2,2,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,All Educators,...,ALL,ALL,ALL,25,"48,982.63","35,892.53","13,090.1",NaN,20,46
3,3,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,Regular,...,Instructional Support,"Supervisor, Instructional",ALL,3,"115,362.61","64,883.95","50,478.65",NaN,12,53
4,4,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,All Educators,...,ALL,ALL,ALL,46,"69,886.27","44,649.62","25,236.64",NaN,19,48


In [35]:
salary_raw[(salary_raw['District Code'] == 10) & (salary_raw['School Year'] == 2025)].head()

,Unnamed: 0,School Year,District Code,District,School Code,Organization,Race,Gender,Grade,SpecialDemo,...,Staff Category,Job Classification,Experience,Educators (FTE),Average Total Salary,Average State Salary,Average Local Salary,Average Federal Salary,Average Years of Experience,Average Years of Age
1244052,2400669,2025,10,Caesar Rodney School District,0,Caesar Rodney School District,Hispanic/Latino,All Educators,All Educators,Regular,...,Official/Administrative,Principal,ALL,1,"129,087.92","77,699.17999999999","51,388.74",NaN,16,44
1244053,2400670,2025,10,Caesar Rodney School District,0,Caesar Rodney School District,Hispanic/Latino,All Educators,All Educators,Regular,...,ALL,ALL,ALL,27,"66,638.99000000001","49,226.92","20,211.54","50,980.37",11,40
1244054,2400671,2025,10,Caesar Rodney School District,0,Caesar Rodney School District,Hispanic/Latino,All Educators,All Educators,All Educators,...,Skilled & Service Workers,Teaching & Clerical Aide,ALL,13,"40,757.78","33,573.5","7,184.28",NaN,11,43
1244055,2400672,2025,10,Caesar Rodney School District,0,Caesar Rodney School District,Hispanic/Latino,All Educators,All Educators,All Educators,...,Skilled & Service Workers,Crafts & Trades,ALL,1,"61,536.02","38,842.7","22,693.32",NaN,17,55
1244056,2400673,2025,10,Caesar Rodney School District,0,Caesar Rodney School District,Hispanic/Latino,All Educators,All Educators,All Educators,...,Classroom Teacher,"Teacher, Special Secondary",ALL,1,"52,385.06",NaN,NaN,"52,385.06",3,31


In [36]:
salary_dropped = salary_raw.drop(columns=['Average State Salary', 'Average Local Salary', 'Average Federal Salary'])
salary_dropped.head()

,Unnamed: 0,School Year,District Code,District,School Code,Organization,Race,Gender,Grade,SpecialDemo,Geography,SubGroup,Staff Type,Staff Category,Job Classification,Experience,Educators (FTE),Average Total Salary,Average Years of Experience,Average Years of Age
0,0,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,Regular,All Educators,White/Regular School,Professional,Official/Administrative,ALL,ALL,14,"95,625.22",19,53
1,1,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,Regular,All Educators,White/Regular School,Professional,Pupil Support,ALL,ALL,1,"97,917.55",1,33
2,2,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,All Educators,All Educators,White,Non-Professional,ALL,ALL,ALL,25,"48,982.63",20,46
3,3,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,Regular,All Educators,White/Regular School,Professional,Instructional Support,"Supervisor, Instructional",ALL,3,"115,362.61",12,53
4,4,2020,24,Smyrna School District,697,Smyrna Administrative Office,White,All Educators,All Educators,All Educators,All Educators,White,ALL,ALL,ALL,ALL,46,"69,886.27",19,48


In [37]:
salary_dropped['Average Total Salary'] = pd.to_numeric(
    salary_dropped['Average Total Salary'].astype(str).str.replace(',', '').str.replace('$', ''),
    errors='coerce'
)
salary_dropped['Average Years of Experience'] = pd.to_numeric(
    salary_dropped['Average Years of Experience'],
    errors='coerce'
)

def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[col] >= lower) & (df[col] <= upper)]

salary_clean = remove_outliers(salary_dropped, 'Average Total Salary')

salary_district = salary_clean.groupby(['School Year', 'District Code']).mean(numeric_only=True).reset_index()
mobility_district = mobility_raw.groupby(['School Year', 'District Code']).mean(numeric_only=True).reset_index()

df_merged = pd.merge(
    salary_district,
    mobility_district,
    on=['School Year', 'District Code'],
    how='inner'
)

In [38]:
target_col = 'Turnover Rate'

cr_2025_data = df_merged[(df_merged['District Code'] == 10) & (df_merged['School Year'] == 2025)]

train_data = df_merged.drop(cr_2025_data.index)

train_data = remove_outliers(train_data, 'Turnover Rate')

X = train_data.drop(columns=[target_col, 'School Year', 'District Code'])
y = train_data[target_col]

X_cr_2025 = cr_2025_data.drop(columns=[target_col, 'School Year', 'District Code'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (165, 9)
Testing data shape: (42, 9)


In [39]:
models = {
    'KNN': {
        'model': KNeighborsRegressor(),
        'params': {'regressor__n_neighbors': [3, 5, 7, 10],
                   'regressor__weights': ['uniform', 'distance']}
    },
    'Decision Tree': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {'regressor__max_depth': [3, 5, 10, None],
                   'regressor__min_samples_split': [2, 5, 10]}
    },
    'Ridge': {
        'model': Ridge(),
        'params': {'regressor__alpha': [0.1, 1.0, 10.0, 100.0]}
    },
    'Lasso': {
        'model': Lasso(),
        'params': {'regressor__alpha': [0.01, 0.1, 1.0, 10.0]}
    }
}

best_models = {}

for name, config in models.items():

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', config['model'])
    ])

    grid = GridSearchCV(
        pipeline,
        config['params'],
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    print(f"--- {name} ---")
    print(f"Best Params: {grid.best_params_}")
    print(f"Best CV MSE: {-grid.best_score_:.4f}\n")

--- KNN ---
Best Params: {'regressor__n_neighbors': 5, 'regressor__weights': 'distance'}
Best CV MSE: 43.0177

--- Decision Tree ---
Best Params: {'regressor__max_depth': 10, 'regressor__min_samples_split': 10}
Best CV MSE: 25.8295

--- Ridge ---
Best Params: {'regressor__alpha': 0.1}
Best CV MSE: 1.0106

--- Lasso ---
Best Params: {'regressor__alpha': 0.01}
Best CV MSE: 1.0108



In [41]:
from sklearn.metrics import r2_score, root_mean_squared_error

winning_model = best_models['Ridge']

test_predictions = winning_model.predict(X_test)
r2 = r2_score(y_test, test_predictions)
rmse = root_mean_squared_error(y_test, test_predictions)

print(f"--- MODEL ACCURACY METRICS ---")
print(f"R-Squared Score: {r2 * 100:.1f}%")
print(f"Average Margin of Error (RMSE): {rmse:.2f} percentage points\n")

X_cr_2025_filled = X_cr_2025.fillna(X_train.mean())

final_prediction = winning_model.predict(X_cr_2025_filled)
print(f"🌟 OFFICIAL PREDICTION 🌟")
print(f"Predicted 2025 Caesar Rodney Turnover Rate: {final_prediction[0]:.2f}%")

--- MODEL ACCURACY METRICS ---
R-Squared Score: 98.9%
Average Margin of Error (RMSE): 0.99 percentage points

🌟 OFFICIAL PREDICTION 🌟
Predicted 2025 Caesar Rodney Turnover Rate: 22.77%
